In [ ]:
from dotenv import load_dotenv
load_dotenv()

from typing_extensions import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver  #store conversation in memory checkpoints (snapshots dello stato). It uses RAM by default (in production si usa sqllite/redis/postgredql/s3)!

from langgraph.checkpoint.sqlite import SqliteSaver  #x persistence su SQLite (db locale file-based)
from langgraph.checkpoint.postgres import PostgresSaver  #persistence su 🔥PostgreSQL

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.prebuilt import ToolNode
import os

# Configuration
BASE_URL = "http://localhost:11434"
MODEL_NAME = "qwen3"

llm = ChatOllama(
    model=MODEL_NAME, 
    base_url=BASE_URL)

In [ ]:
import sys  #python system module, for manipolare il path di ricerca dei moduli
sys.path.append("../05. LangGraph ReAct Agent with Tools")  #aggiunge cartella al python import path, fa un "python, cerca moduli anche qui"
import my_tools  ##in realta è il code sopra sopra in alto ad inizio di file '4..'

# my_tools.get_weather.invoke({'location': "Mumbai"})
my_tools.calculate.invoke({'expression': '2+2*1.4/23-34'}) #esegui tool manualmente, passandogli argomenti richiesti
all_tools = [my_tools.get_weather, my_tools.calculate]  #lista tools disponibili per agente

class AgentState(TypedDict):
    messages: Annotated[list, operator.add]

In [ ]:
def agent_node(state: AgentState):
    llm_with_tools = llm.bind_tools(all_tools)
    system_message = SystemMessage(
        """You are a friendly assistant with memory. 
           Use the availale tools to help the user when needed.                   
           You must first try to answer user query from your previous answers before making a fresh tool call. Do not make answers by yourself if you are not sure.""")
    messages = [system_message] + state['messages']  #aggiungi messaggi precedenti (memoria) a system message, che da istruzioni al llm su come comportarsi.
    #e.g. state['messages'] è 
    #[
    #   HumanMessage("Hi")
    #]
    #che diventa quindi 
    #[
    #   SystemMessage(...),
    #   HumanMessage("Hi")
    #]
    #🔥in questo il modello vede tutta la conversazione precedente (memoria) e può decidere se rispondere o se chiamare un tool per rispondere alla query dell'utente, in base a quello che ha visto nella conversazione precedente (memoria)
    response = llm_with_tools.invoke(messages)
    if hasattr(response, 'tool_calls') and response.tool_calls:
        for tc in response.tool_calls:
            print(f"[AGENT] called Tool {tc.get('name', '?')} with args {tc.get('args', '?')}")
    else:
        print(f"[AGENT] Responding...")
    return {'messages': [response]}

state = {"messages": [HumanMessage("Hi")]}
result = agent_node(state)
result

In [ ]:
def should_continue(state: AgentState):
    last = state['messages'][-1]
    if hasattr(last, 'tool_calls') and last.tool_calls:
        return "tools"
    else:
        return END

In [ ]:
def create_agent(checkpointer):
    builder = StateGraph(AgentState)
    builder.add_node("agent", agent_node)
    builder.add_node("tools", ToolNode(all_tools))
    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", should_continue, ["tools", END])  #conditional edge
    builder.add_edge("tools", "agent")
    graph = builder.compile(checkpointer=checkpointer)  #here setto anche il checkpointer
    return graph

checkpointer = MemorySaver()  #🔥memoria persistente del graph, checkpoint è lo snapshot dello stato!
agent = create_agent(checkpointer)
agent

In [ ]:

import sqlite3  #modulo python x db Sqlite, che è un db file-based, quindi salva i dati su file locale (db/checkpoints.db in questo caso)
import os  #modulo python x filesystem

os.makedirs('db', exist_ok=True)  #crea cartella 'db' se non esiste già (exist_ok=True evita errore se esiste gia la cartella), dove salveremo il file del db Sqlite (checkpoints.db)
db_path = "db/checkpoints.db"
conn = sqlite3.connect(
    db_path, 
    check_same_thread=False) #connessione al db Sqlite, se il file non esiste lo crea, check_same_thread=False permette multi-thread access (importante xk langgraph puo usare async/threads/concurrency/ect)
checkpointer = SqliteSaver(conn)  #update var, ORA NON E' PIU SU RAM only e volatile, MA DIVENTA PERSISTENT ON DISK GRAZIE A SQLLITE!🔥
#🔥🔥QUESTO checkpointer POI LO PASSI AL FUNZIONE DEL GRAPH, che lo finisce di settare con graph = builder.compile(checkpointer=checkpointer)

In [ ]:
def chat(agent, query, thread_id):  #passi anche agent come parametro, così puoi usare la stessa funzione x diversi agenti (es. con checkpointer diversi, o anche agent diversi, es. con tool diversi)
    config = {"configurable": {"thread_id": thread_id}}  #session/thread id, per distinguere chat diverse
    for chunk in agent.stream({'messages': [query]}, config=config):  #.strem() per vedere tool calls/tool outputs/reasoning steps/ect mentre avvengono. invece .invoke() aspetta la risposta finale, e non mostra quello che succede durante il processo. langgraph converte automaticamente query->HumanMessage(query)
        if 'agent' in chunk:  #perche con .stream(), ottieni pezzi e.g. {"agent":{...}} or  {"tools":{...}} ect
            chunk = chunk.get('agent')  #ora chunk contiene la risposta dell'agente, che può essere una risposta finale (se l'agente ha deciso di rispondere direttamente) o una tool call (se l'agente ha deciso di chiamare un tool per rispondere alla query dell'utente).
        else:
            chunk = chunk.get('tools')
        if hasattr(chunk, 'tool_calls') and chunk.tool_calls:
            for tc in chunk.tool_calls:
                print(f"[AGENT] called Tool {tc.get('name', '?')} with args {tc.get('args', '?')}")
        else:
            print(f"[AGENT/ToolMessage] Responding.\n{chunk['messages'][0].content}")
        # print(chunk)   

agent = create_agent(checkpointer)
agent  

query = "What is the current weather in New Delhi?"
chat(agent, query, "laxmikant")

In [ ]:
import psycopg  #postgresql moderno per python, più facile da usare di psycopg2, supporta async, ect
conn = psycopg.connect(
    os.getenv("POSTGRESQL_URL"),  #read env var 
    autocommit=True,  #ogni query viene salvata immediatamente (senza dovresti altrimenti fare sempre conn.commit() manualmente dopo ogni query)
    prepare_threshold=0)  #disabilita prepared statements, per evitare problemi con query dinamiche (es. quelle generate da langgraph)(quindi ora hai migliore compatiibilita con pooling/langgraph checkpointing/better x alcuni deployment postgresql), in produzione puoi valutare se tenerlo abilitato o no in base alle tue esigenze di performance e sicurezza
    #🔥connessione postgresql 
checkpointer = PostgresSaver(conn)  #update var
checkpointer.setup()  #crea tabelle necessarie in postgresql per salvare checkpoint, 🔥da eseguire solo la prima volta, poi puoi commentare questa riga o rimuoverla, altrimenti ogni volta che esegui lo script cercherà di creare le tabelle e ti darà errore se esistono già. In produzione puoi gestire migrazioni e setup del db in modo più robusto con strumenti come Alembic o Flyway.

In [ ]:

agent = create_agent(checkpointer)
query = "What is the current weather in New Delhi?"
chat(agent, query, "laxmikant")

In [ ]:
with PostgresSaver.from_conn_string(os.getenv("POSTGRESQL_URL")) as checkpointer:  #crea connessione postgresql & checpointer langgraph
    agent = create_agent(checkpointer)
    query = "what is 2 + 2?"
    chat(agent, query, "laxmikant")
#close db connection automatically with context manager (with statement) (E' COME LA CONNECTION STRING X ADO.NET (aspnetcore) CHE LA FAI CHE SI CHIUDE SEMPRE AUTOMATICAMENTE!)

with PostgresSaver.from_conn_string(os.getenv("POSTGRESQL_URL")) as checkpointer:
    agent = create_agent(checkpointer)
    query = "How many questions I have asked previosly? tell me all."
    chat(agent, query, "laxmikant")